# SECOM 전처리 — 1주차

`01_eda.ipynb`에서 확인한 내용을 반영해 모델 학습용 데이터를 만든다.

1. 결측률 50% 초과 피처 제거
2. Train / Validation / Test 3분할 (test는 임계값 튜닝에 전혀 쓰지 않는 최종 홀드아웃)
3. 남은 결측값 중앙값 대체
4. 분산 0(저분산) 피처 제거
5. `StandardScaler` 정규화
6. 클래스 불균형 처리(SMOTE) — train에만 적용
7. `data/processed/`에 저장

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

DATA_DIR = "../data/raw"
OUT_DIR = "../data/processed"
RANDOM_STATE = 42

## 1. 데이터 로드

In [2]:
X = pd.read_csv(f"{DATA_DIR}/secom.data", sep=" ", header=None)
X.columns = [f"feature_{i+1}" for i in range(X.shape[1])]

labels_raw = pd.read_csv(f"{DATA_DIR}/secom_labels.data", sep=" ", header=None, names=["label", "timestamp"])
y = (labels_raw["label"] == 1).astype(int)  # 1=불량(Fail), 0=정상(Pass)

print(X.shape, y.value_counts().to_dict())

(1567, 590) {0: 1463, 1: 104}


## 2. 결측률 50% 초과 피처 제거

In [3]:
missing_ratio = X.isna().mean()
high_missing_cols = missing_ratio[missing_ratio > 0.5].index.tolist()
X = X.drop(columns=high_missing_cols)
print(f"제거된 피처 수: {len(high_missing_cols)} -> 남은 피처 수: {X.shape[1]}")

제거된 피처 수: 28 -> 남은 피처 수: 562


## 3. Train / Validation / Test 분리 (64% / 16% / 20%)

- **Test**: 최종 성능 보고용, 어떤 튜닝에도 쓰지 않는다.
- **Validation**: XGBoost 분류 임계값 튜닝용 (실제 클래스 비율 유지, SMOTE 미적용).
- **Train**: 모델 학습용. 지도학습(XGBoost)에는 SMOTE 적용본을, 비지도(Isolation Forest)에는 원본 비율본을 사용.

In [4]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=RANDOM_STATE
)
print("train:", X_train.shape, y_train.value_counts().to_dict())
print("val:  ", X_val.shape, y_val.value_counts().to_dict())
print("test: ", X_test.shape, y_test.value_counts().to_dict())

train: (1002, 562) {0: 936, 1: 66}
val:   (251, 562) {0: 234, 1: 17}
test:  (314, 562) {0: 293, 1: 21}


## 4. 결측값 대체 (중앙값, train 기준 fit)

In [5]:
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_imp = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

## 5. 저분산(분산 0) 피처 제거 (train 기준)

In [6]:
variances = X_train_imp.var()
zero_var_cols = variances[variances == 0].index.tolist()
X_train_imp = X_train_imp.drop(columns=zero_var_cols)
X_val_imp = X_val_imp.drop(columns=zero_var_cols)
X_test_imp = X_test_imp.drop(columns=zero_var_cols)
print(f"제거된 저분산 피처 수: {len(zero_var_cols)} -> 남은 피처 수: {X_train_imp.shape[1]}")

제거된 저분산 피처 수: 122 -> 남은 피처 수: 440


## 6. StandardScaler 정규화 (train 기준 fit)

In [7]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train_imp.columns, index=X_train_imp.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val_imp), columns=X_val_imp.columns, index=X_val_imp.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imp), columns=X_test_imp.columns, index=X_test_imp.index)

## 7. 클래스 불균형 처리 (SMOTE, train만)

In [8]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print("SMOTE 적용 전:", y_train.value_counts().to_dict())
print("SMOTE 적용 후:", y_train_res.value_counts().to_dict())

SMOTE 적용 전: {0: 936, 1: 66}
SMOTE 적용 후: {0: 936, 1: 936}


## 8. 저장

- `X_train.csv` / `y_train.csv`: SMOTE 적용된 학습 데이터 (XGBoost용)
- `X_train_raw.csv` / `y_train_raw.csv`: SMOTE 미적용 원본 비율 학습 데이터 (Isolation Forest용)
- `X_val.csv` / `y_val.csv`: 검증 데이터 (원본 비율, 분류 임계값 튜닝용)
- `X_test.csv` / `y_test.csv`: 테스트 데이터 (원본 비율, 최종 성능 평가 전용 — 어떤 튜닝에도 사용 금지)

In [9]:
import os
os.makedirs(OUT_DIR, exist_ok=True)

X_train_res.to_csv(f"{OUT_DIR}/X_train.csv", index=False)
y_train_res.to_csv(f"{OUT_DIR}/y_train.csv", index=False)

X_train_scaled.to_csv(f"{OUT_DIR}/X_train_raw.csv", index=False)
y_train.to_csv(f"{OUT_DIR}/y_train_raw.csv", index=False)

X_val_scaled.to_csv(f"{OUT_DIR}/X_val.csv", index=False)
y_val.to_csv(f"{OUT_DIR}/y_val.csv", index=False)

X_test_scaled.to_csv(f"{OUT_DIR}/X_test.csv", index=False)
y_test.to_csv(f"{OUT_DIR}/y_test.csv", index=False)

print("저장 완료:", sorted(os.listdir(OUT_DIR)))

저장 완료: ['.gitkeep', 'X_test.csv', 'X_train.csv', 'X_train_raw.csv', 'X_val.csv', 'y_test.csv', 'y_train.csv', 'y_train_raw.csv', 'y_val.csv']


## 다음 단계

- `03_modeling.ipynb`: Isolation Forest(비지도, `X_train_raw`로 학습) + XGBoost(지도, `X_train`으로 학습, `X_val`로 임계값 튜닝) 앙상블, `X_test`로 최종 평가